# 03 · EDA, Analisis Temporal-Spasial, dan Insight

Notebook ini menjawab lima pertanyaan analitik tentang cuaca di 38 ibu kota provinsi Indonesia (2020–2024):

1. **Spasial**: provinsi mana yang paling basah dan paling kering, dan seberapa besar perbedaannya?
2. **Musiman**: apakah semua wilayah punya pola musim hujan–kemarau yang sama?
3. **Antartahun**: adakah tahun yang menyimpang, dan wilayah mana yang paling terdampak?
4. **Harian (diurnal)**: pada jam berapa hujan paling sering turun?
5. **Suhu**: bagaimana tren suhu dan apa yang menentukan perbedaan suhu antarprovinsi?

Pembagian peran: **DuckDB** mengerjakan agregasi SQL langsung di atas file Parquet hasil notebook 02,
sedangkan **Polars** dipakai untuk transformasi lanjutan (pivot, window function, korelasi).
Semua visualisasi dibuat dengan **Plotly** (interaktif: hover, zoom, klik legenda untuk menyembunyikan seri)
dan disimpan sebagai HTML di `output/figures/`.

> Jalankan notebook 02 terlebih dahulu agar `data/processed/cuaca_bersih.parquet` tersedia.

In [1]:
import sys
from pathlib import Path

# Notebook dapat dijalankan dari folder notebooks/ maupun dari root repository.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import duckdb
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import polars as pl

from src.config import CLEAN_PARQUET, URUTAN_PULAU
from src.viz import DIVERGEN_HUJAN, DIVERGEN_SUHU, SEKUENSIAL, WARNA_PULAU, simpan

pio.renderers.default = "plotly_mimetype+notebook_connected"
pl.Config.set_tbl_rows(40)
pl.Config.set_float_precision(2)

assert CLEAN_PARQUET.exists(), "Jalankan notebook 02_data_cleaning terlebih dahulu"
PQ = CLEAN_PARQUET.as_posix()
con = duckdb.connect()
con.sql(f"CREATE VIEW cuaca AS SELECT * FROM read_parquet('{PQ}')")
con.sql("SELECT COUNT(*) AS n_baris, COUNT(DISTINCT provinsi) AS n_provinsi FROM cuaca").pl()

n_baris,n_provinsi
i64,i64
1666224,38


## 1. Analisis spasial: curah hujan tahunan per provinsi

Curah hujan tahunan dihitung sebagai jumlah presipitasi per provinsi per tahun, lalu dirata-ratakan
selama 5 tahun.

In [2]:
hujan_tahunan = con.sql("""
    WITH per_tahun AS (
        SELECT provinsi, kota, pulau, tahun,
               ANY_VALUE(latitude)  AS lat,
               ANY_VALUE(longitude) AS lon,
               SUM(precipitation)   AS mm
        FROM cuaca
        GROUP BY ALL
    )
    SELECT provinsi, kota, pulau, ANY_VALUE(lat) AS lat, ANY_VALUE(lon) AS lon,
           ROUND(AVG(mm), 0) AS mm_per_tahun
    FROM per_tahun
    GROUP BY provinsi, kota, pulau
    ORDER BY mm_per_tahun DESC
""").pl()

terbasah, terkering = hujan_tahunan.row(0, named=True), hujan_tahunan.row(-1, named=True)
print(f"Terbasah : {terbasah['provinsi']} ({terbasah['kota']}) {terbasah['mm_per_tahun']:,.0f} mm/tahun")
print(f"Terkering: {terkering['provinsi']} ({terkering['kota']}) {terkering['mm_per_tahun']:,.0f} mm/tahun")
print(f"Rasio    : {terbasah['mm_per_tahun'] / terkering['mm_per_tahun']:.1f}x")
pl.concat([hujan_tahunan.head(5), hujan_tahunan.tail(5)]).select("provinsi", "kota", "pulau", "mm_per_tahun")

Terbasah : Maluku Utara (Sofifi) 4,903 mm/tahun
Terkering: Nusa Tenggara Timur (Kupang) 1,407 mm/tahun
Rasio    : 3.5x


provinsi,kota,pulau,mm_per_tahun
str,str,str,f64
"""Maluku Utara""","""Sofifi""","""Maluku""",4903.00
"""Papua Tengah""","""Nabire""","""Papua""",4720.00
"""Sulawesi Barat""","""Mamuju""","""Sulawesi""",4332.00
"""Sumatera Barat""","""Padang""","""Sumatera""",4317.00
"""Papua Barat Daya""","""Sorong""","""Papua""",4255.00
"""Banten""","""Serang""","""Jawa""",1990.00
"""Papua Selatan""","""Merauke""","""Papua""",1972.00
"""Jawa Timur""","""Surabaya""","""Jawa""",1894.00
"""Nusa Tenggara Barat""","""Mataram""","""Bali-Nusa Tenggara""",1712.00


In [3]:
# Visualisasi 1: peta curah hujan tahunan (warna & ukuran = mm/tahun)
fig1 = go.Figure(
    go.Scattergeo(
        lat=hujan_tahunan["lat"],
        lon=hujan_tahunan["lon"],
        text=hujan_tahunan["provinsi"],
        customdata=hujan_tahunan.select("kota", "mm_per_tahun").to_numpy(),
        mode="markers",
        marker=dict(
            size=(hujan_tahunan["mm_per_tahun"] / 180).to_list(),
            color=hujan_tahunan["mm_per_tahun"],
            colorscale=SEKUENSIAL,
            line=dict(width=1, color="white"),
            colorbar=dict(title="mm/tahun"),
        ),
        hovertemplate="<b>%{text}</b><br>%{customdata[0]}<br>%{customdata[1]:,.0f} mm/tahun<extra></extra>",
    )
)
fig1.update_geos(
    lonaxis_range=[94, 142], lataxis_range=[-11.5, 7.5],
    showcountries=True, countrycolor="#c3c2b7", showland=True, landcolor="#f0efec",
    showocean=True, oceancolor="#ffffff", projection_type="mercator",
)
fig1.update_layout(
    title="Rata-rata curah hujan tahunan 2020–2024 per ibu kota provinsi<br>"
          "<sup>Bagian tenggara (Nusa Tenggara, Jawa Timur) jauh lebih kering daripada Maluku, Papua, dan Sulawesi</sup>",
    height=520,
)
simpan(fig1, "01_peta_curah_hujan_tahunan")
fig1.show()

## 2. Pola musiman per pulau

Untuk membandingkan pola musim antarwilayah yang total hujannya berbeda, hujan setiap bulan dinyatakan
sebagai **persentase dari total hujan tahunan** pulau tersebut. Bila hujan merata sepanjang tahun, setiap
bulan bernilai sekitar 8,3%.

In [4]:
musiman = con.sql("""
    SELECT pulau, bulan, SUM(precipitation) AS mm
    FROM cuaca
    GROUP BY pulau, bulan
""").pl().with_columns(
    (pl.col("mm") / pl.col("mm").sum().over("pulau") * 100).round(1).alias("persen")
)
pivot_musim = (
    musiman.pivot(on="bulan", index="pulau", values="persen", sort_columns=True)
    .with_columns(pl.col("pulau").cast(pl.Enum(URUTAN_PULAU)))
    .sort("pulau")
)
# Porsi hujan Nov–Mar dan Jul–Agu per pulau
porsi = musiman.group_by("pulau").agg(
    pl.col("persen").filter(pl.col("bulan").is_in([11, 12, 1, 2, 3])).sum().alias("persen_nov_mar"),
    pl.col("persen").filter(pl.col("bulan").is_in([7, 8])).sum().alias("persen_jul_agu"),
    pl.col("bulan").sort_by("persen").last().alias("bulan_puncak"),
).with_columns(pl.col("pulau").cast(pl.Enum(URUTAN_PULAU))).sort("pulau")
porsi

pulau,persen_nov_mar,persen_jul_agu,bulan_puncak
enum,f64,f64,i8
"""Sumatera""",44.20,13.10,11
"""Jawa""",64.90,4.70,12
"""Bali-Nusa Tenggara""",75.70,2.80,12
"""Kalimantan""",49.00,12.10,11
"""Sulawesi""",42.00,15.80,11
"""Maluku""",38.40,17.10,5
"""Papua""",47.40,13.80,3


In [5]:
# Visualisasi 2: heatmap persentase hujan bulanan per pulau
BULAN = ["Jan", "Feb", "Mar", "Apr", "Mei", "Jun", "Jul", "Agu", "Sep", "Okt", "Nov", "Des"]
z = pivot_musim.drop("pulau").to_numpy()
fig2 = go.Figure(
    go.Heatmap(
        z=z, x=BULAN, y=pivot_musim["pulau"].cast(pl.String).to_list(),
        colorscale=SEKUENSIAL, zmin=0, xgap=2, ygap=2,
        text=z, texttemplate="%{text:.0f}", textfont=dict(size=11),
        colorbar=dict(title="% hujan<br>tahunan"),
        hovertemplate="<b>%{y}</b> · %{x}<br>%{z:.1f}% dari hujan tahunan<extra></extra>",
    )
)
fig2.update_layout(
    title="Persentase hujan tahunan yang turun di setiap bulan<br>"
          "<sup>Jawa dan Bali–Nusa Tenggara bermusim tegas; Maluku justru basah di pertengahan tahun</sup>",
    yaxis=dict(autorange="reversed"), height=430,
)
simpan(fig2, "02_heatmap_pola_musiman")
fig2.show()

## 3. Variasi antartahun

### 3a. Deret waktu curah hujan bulanan per pulau

In [6]:
bulanan = con.sql("""
    SELECT pulau, DATE_TRUNC('month', time) AS bulan,
           SUM(precipitation) / COUNT(DISTINCT provinsi) AS mm_per_provinsi
    FROM cuaca
    GROUP BY ALL
    ORDER BY bulan
""").pl()

# Visualisasi 3: small multiples, satu panel per pulau dengan sumbu y yang sama
# (7 garis dalam satu panel terlalu padat untuk dibaca)
fig3 = px.line(
    bulanan.with_columns(pl.col("mm_per_provinsi").round(0)),
    x="bulan", y="mm_per_provinsi", color="pulau", facet_row="pulau",
    category_orders={"pulau": URUTAN_PULAU}, color_discrete_map=WARNA_PULAU,
    labels={"bulan": "", "mm_per_provinsi": "mm", "pulau": "Pulau"},
)
fig3.update_traces(line=dict(width=2), hovertemplate="%{x|%b %Y}: %{y:.0f} mm")
fig3.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1], textangle=0, x=0.01, xanchor="left", y=a.y + 0.045))
fig3.add_vrect(x0="2023-07-01", x1="2023-10-31", fillcolor="#e34948", opacity=0.10, line_width=0,
               row="all", col=1)
fig3.update_yaxes(title_text="", tickvals=[0, 200, 400])
fig3.update_layout(
    title="Curah hujan bulanan per pulau, 2020–2024 (mm, rata-rata provinsi)<br>"
          "<sup>Area merah = Juli–Oktober 2023, musim kemarau terkering selama periode data</sup>",
    showlegend=False, height=950, margin=dict(t=110),
)
simpan(fig3, "03_line_hujan_bulanan_per_pulau")
fig3.show()

### 3b. Anomali curah hujan tahunan per provinsi

Anomali = (hujan tahun tersebut − rata-rata 5 tahun provinsi itu) ÷ rata-rata 5 tahun × 100%.
Nilai negatif (merah) berarti lebih kering dari biasanya.

In [7]:
anomali = (
    con.sql("SELECT provinsi, pulau, tahun, SUM(precipitation) AS mm FROM cuaca GROUP BY ALL").pl()
    .with_columns(((pl.col("mm") / pl.col("mm").mean().over("provinsi") - 1) * 100).round(1).alias("anomali_persen"))
)
ringkas_tahun = anomali.group_by("tahun").agg(
    pl.col("anomali_persen").median().alias("median_anomali_persen"),
    (pl.col("anomali_persen") < 0).sum().alias("n_provinsi_lebih_kering"),
).sort("tahun")
nasional = con.sql("""
    SELECT tahun, SUM(precipitation) / 38 AS mm_per_provinsi FROM cuaca GROUP BY tahun ORDER BY tahun
""").pl()
ringkas_tahun.join(nasional, on="tahun")

tahun,median_anomali_persen,n_provinsi_lebih_kering,mm_per_provinsi
i16,f64,u32,f64
2020,2.00,16,3155.11
2021,2.35,12,3168.58
2022,9.45,7,3290.21
2023,-19.45,36,2541.83
2024,2.70,15,3164.51


In [8]:
# Visualisasi 4: heatmap anomali (divergen merah-abu-biru, titik tengah 0%)
urut_prov = (
    anomali.with_columns(pl.col("pulau").cast(pl.Enum(URUTAN_PULAU)))
    .sort("pulau", "provinsi").get_column("provinsi").unique(maintain_order=True).to_list()
)
mat = anomali.pivot(on="tahun", index="provinsi", values="anomali_persen", sort_columns=True)
mat = pl.DataFrame({"provinsi": urut_prov}).join(mat, on="provinsi", how="left")
z = mat.drop("provinsi").to_numpy()
batas = float(abs(z).max())
fig4 = go.Figure(
    go.Heatmap(
        z=z, x=[str(c) for c in mat.columns[1:]], y=mat["provinsi"].to_list(),
        colorscale=DIVERGEN_HUJAN, zmid=0, zmin=-batas, zmax=batas, xgap=2, ygap=1,
        colorbar=dict(title="Anomali<br>(%)"),
        hovertemplate="<b>%{y}</b> · %{x}<br>%{z:+.1f}% dari rata-rata<extra></extra>",
    )
)
fig4.update_layout(
    title="Anomali curah hujan tahunan per provinsi (dikelompokkan per pulau)<br>"
          "<sup>2023 merah di hampir semua provinsi; 2022 cenderung lebih basah</sup>",
    yaxis=dict(autorange="reversed", tickfont=dict(size=10)), height=820,
)
simpan(fig4, "04_heatmap_anomali_hujan")
fig4.show()

In [9]:
# Fokus: hujan musim kemarau (Juli–Oktober) di Jawa per tahun
kemarau_jawa = con.sql("""
    SELECT tahun, ROUND(SUM(precipitation) / COUNT(DISTINCT provinsi), 0) AS mm_jul_okt
    FROM cuaca
    WHERE pulau = 'Jawa' AND bulan BETWEEN 7 AND 10
    GROUP BY tahun ORDER BY tahun
""").pl()
kemarau_jawa

tahun,mm_jul_okt
i16,f64
2020,400.00
2021,425.00
2022,712.00
2023,139.00
2024,253.00


## 4. Siklus harian (diurnal) hujan

Waktu pada data sudah dalam **jam lokal** (WIB/WITA/WIT), sehingga jam antarpulau dapat dibandingkan langsung.

In [10]:
diurnal = con.sql("""
    SELECT pulau, jam,
           AVG(precipitation)              AS mm_per_jam,
           100 * AVG(jam_hujan::INT)       AS peluang_hujan_persen
    FROM cuaca
    GROUP BY pulau, jam
    ORDER BY pulau, jam
""").pl()

puncak = (
    diurnal.sort("mm_per_jam", descending=True).group_by("pulau").first()
    .join(diurnal.group_by("pulau").agg(pl.col("mm_per_jam").min().alias("minimum")), on="pulau")
    .with_columns((pl.col("mm_per_jam") / pl.col("minimum")).round(1).alias("rasio_puncak_vs_minimum"))
    .with_columns(pl.col("pulau").cast(pl.Enum(URUTAN_PULAU))).sort("pulau")
)
siang = con.sql("SELECT 100 * SUM(precipitation) FILTER (WHERE jam BETWEEN 12 AND 17) / SUM(precipitation) AS persen FROM cuaca").fetchone()[0]
print(f"Porsi hujan yang turun pukul 12.00–17.59 waktu lokal: {siang:.1f}% (6 dari 24 jam = 25%)")
puncak.select("pulau", pl.col("jam").alias("jam_puncak"), pl.col("mm_per_jam").round(2), "rasio_puncak_vs_minimum")

Porsi hujan yang turun pukul 12.00–17.59 waktu lokal: 51.2% (6 dari 24 jam = 25%)


pulau,jam_puncak,mm_per_jam,rasio_puncak_vs_minimum
enum,i8,f64,f64
"""Sumatera""",15,0.79,4.20
"""Jawa""",15,0.75,7.10
"""Bali-Nusa Tenggara""",15,0.57,5.80
"""Kalimantan""",15,0.87,5.00
"""Sulawesi""",14,1.40,9.20
"""Maluku""",15,1.27,6.90
"""Papua""",14,0.88,3.80


In [11]:
# Visualisasi 5: siklus diurnal
fig5 = px.line(
    diurnal, x="jam", y="mm_per_jam", color="pulau", markers=True,
    category_orders={"pulau": URUTAN_PULAU}, color_discrete_map=WARNA_PULAU,
    labels={"jam": "Jam lokal", "mm_per_jam": "Rata-rata hujan (mm/jam)", "pulau": "Pulau"},
)
fig5.update_traces(line=dict(width=2), marker=dict(size=7), hovertemplate="%{y:.2f} mm/jam")
fig5.add_vrect(x0=12, x1=17, fillcolor="#2a78d6", opacity=0.06, line_width=0,
               annotation_text="12.00–17.00", annotation_position="top left")
fig5.update_layout(
    title="Siklus harian hujan: rata-rata presipitasi per jam lokal<br>"
          "<sup>Di semua pulau hujan memuncak pukul 14–15 siang dan paling sedikit menjelang pagi</sup>",
    xaxis=dict(dtick=2), hovermode="x unified", height=460,
)
simpan(fig5, "05_line_siklus_diurnal")
fig5.show()

## 5. Suhu

### 5a. Anomali suhu bulanan nasional

Anomali suhu = suhu rata-rata bulan tersebut − rata-rata bulan kalender yang sama selama 2020–2024,
sehingga pengaruh musim dihilangkan.

In [12]:
suhu_bulanan = (
    pl.scan_parquet(CLEAN_PARQUET)
    .group_by(pl.col("time").dt.truncate("1mo").alias("bulan_tahun"), "bulan")
    .agg(pl.col("temperature_2m").mean().alias("suhu"))
    .with_columns((pl.col("suhu") - pl.col("suhu").mean().over("bulan")).alias("anomali"))
    .sort("bulan_tahun")
    .collect()
)
suhu_tahunan = con.sql("SELECT tahun, ROUND(AVG(temperature_2m), 2) AS suhu_rata2 FROM cuaca GROUP BY tahun ORDER BY tahun").pl()
sejak = suhu_bulanan.filter(pl.col("bulan_tahun") >= pl.datetime(2023, 7, 1))
print(f"Bulan dengan anomali positif sejak Juli 2023: {(sejak['anomali'] > 0).sum()} dari {sejak.height}")
print("Bulan terhangat relatif terhadap normalnya:")
print(suhu_bulanan.sort("anomali", descending=True).head(3).select("bulan_tahun", pl.col("anomali").round(2)))
suhu_tahunan

Bulan dengan anomali positif sejak Juli 2023: 18 dari 18
Bulan terhangat relatif terhadap normalnya:
shape: (3, 2)
┌─────────────────────┬─────────┐
│ bulan_tahun         ┆ anomali │
│ ---                 ┆ ---     │
│ datetime[μs]        ┆ f32     │
╞═════════════════════╪═════════╡
│ 2024-02-01 00:00:00 ┆ 0.54    │
│ 2024-09-01 00:00:00 ┆ 0.52    │
│ 2024-03-01 00:00:00 ┆ 0.51    │
└─────────────────────┴─────────┘


tahun,suhu_rata2
i16,f64
2020,27.22
2021,27.00
2022,26.94
2023,27.27
2024,27.54


In [13]:
# Visualisasi 6: bar anomali suhu bulanan (divergen biru-merah)
fig6 = go.Figure(
    go.Bar(
        x=suhu_bulanan["bulan_tahun"], y=suhu_bulanan["anomali"],
        marker=dict(color=suhu_bulanan["anomali"], colorscale=DIVERGEN_SUHU, cmid=0,
                    colorbar=dict(title="°C")),
        hovertemplate="%{x|%b %Y}<br>%{y:+.2f} °C dari normal<extra></extra>",
    )
)
fig6.update_layout(
    title="Anomali suhu bulanan rata-rata 38 ibu kota provinsi<br>"
          "<sup>Sejak pertengahan 2023 hampir setiap bulan lebih hangat dari normal</sup>",
    yaxis_title="Anomali suhu (°C)", bargap=0.15, height=430,
)
fig6.add_hline(y=0, line_color="#898883", line_width=1)
simpan(fig6, "06_bar_anomali_suhu_bulanan")
fig6.show()

### 5b. Apa yang menentukan suhu rata-rata suatu provinsi?

Hipotesis: di wilayah tropis, perbedaan suhu antarprovinsi lebih ditentukan **ketinggian tempat** daripada
lintang. Data tidak berisi elevasi, tetapi **tekanan udara permukaan** turun sekitar 1 hPa setiap ±8–10 m
kenaikan, sehingga bisa dipakai sebagai proksi ketinggian.

In [14]:
per_prov = (
    pl.scan_parquet(CLEAN_PARQUET)
    .group_by("provinsi", "kota", "pulau")
    .agg(
        pl.col("temperature_2m").mean().alias("suhu"),
        pl.col("surface_pressure").mean().alias("tekanan"),
        pl.col("latitude").first().abs().alias("lintang_abs"),
        pl.col("relative_humidity_2m").mean().alias("kelembapan"),
    )
    .collect()
)
korelasi = per_prov.select(
    pl.corr("suhu", "tekanan").alias("r_suhu_vs_tekanan"),
    pl.corr("suhu", "lintang_abs").alias("r_suhu_vs_lintang"),
)
print(korelasi)
tanpa_tinggi = per_prov.filter(pl.col("tekanan") > 990)
print(f"Rentang suhu semua provinsi       : {per_prov['suhu'].min():.1f}–{per_prov['suhu'].max():.1f} °C")
print(f"Rentang suhu provinsi dataran rendah (tekanan > 990 hPa, n={tanpa_tinggi.height}): "
      f"{tanpa_tinggi['suhu'].min():.1f}–{tanpa_tinggi['suhu'].max():.1f} °C")
per_prov.sort("tekanan").head(4).select("provinsi", "kota", pl.col("suhu").round(1), pl.col("tekanan").round(0))

shape: (1, 2)
┌───────────────────┬───────────────────┐
│ r_suhu_vs_tekanan ┆ r_suhu_vs_lintang │
│ ---               ┆ ---               │
│ f32               ┆ f32               │
╞═══════════════════╪═══════════════════╡
│ 0.95              ┆ -0.08             │
└───────────────────┴───────────────────┘
Rentang suhu semua provinsi       : 20.3–28.5 °C
Rentang suhu provinsi dataran rendah (tekanan > 990 hPa, n=36): 26.3–28.5 °C


provinsi,kota,suhu,tekanan
str,str,f32,f32
"""Papua Pegunungan""","""Wamena""",20.30,840.00
"""Jawa Barat""","""Bandung""",24.10,935.00
"""DI Yogyakarta""","""Yogyakarta""",26.30,999.00
"""Nusa Tenggara Timur""","""Kupang""",27.80,999.00


In [15]:
# Visualisasi 7: scatter suhu vs tekanan permukaan (proksi ketinggian)
fig7 = px.scatter(
    per_prov.with_columns(pl.col("suhu").round(2), pl.col("tekanan").round(1)),
    x="tekanan", y="suhu", color="pulau", hover_name="provinsi", hover_data={"kota": True, "pulau": False},
    category_orders={"pulau": URUTAN_PULAU}, color_discrete_map=WARNA_PULAU,
    labels={"tekanan": "Tekanan permukaan rata-rata (hPa) — makin kecil = makin tinggi", "suhu": "Suhu rata-rata (°C)", "pulau": "Pulau"},
)
fig7.update_traces(marker=dict(size=11, line=dict(width=2, color="white")))
for row in per_prov.sort("tekanan").head(2).iter_rows(named=True):
    fig7.add_annotation(x=row["tekanan"], y=row["suhu"], text=row["kota"], showarrow=True, arrowhead=0, ax=40, ay=-25)
fig7.update_layout(
    title="Suhu rata-rata vs tekanan permukaan per ibu kota provinsi<br>"
          "<sup>Kota dataran tinggi (Wamena, Bandung) jauh lebih sejuk; selebihnya hanya selisih ±2 °C</sup>",
    height=480,
)
simpan(fig7, "07_scatter_suhu_vs_tekanan")
fig7.show()

In [16]:
# Daftar file visualisasi yang tersimpan
from src.config import FIGURES_DIR
sorted(p.name for p in FIGURES_DIR.glob("*.html"))

['01_peta_curah_hujan_tahunan.html',
 '02_heatmap_pola_musiman.html',
 '03_line_hujan_bulanan_per_pulau.html',
 '04_heatmap_anomali_hujan.html',
 '05_line_siklus_diurnal.html',
 '06_bar_anomali_suhu_bulanan.html',
 '07_scatter_suhu_vs_tekanan.html']

## 6. Insight analitik

Semua angka di bawah berasal dari output sel-sel di atas. Catatan: "normal" dan "anomali" di notebook ini
mengacu pada rata-rata 2020–2024, bukan normal klimatologis 30 tahun BMKG, sehingga ukurannya relatif
terhadap periode data.

### Insight 1: Ketimpangan hujan antarwilayah mencapai 3,5 kali lipat
Maluku Utara (Sofifi) menerima rata-rata **4.903 mm/tahun**, sedangkan Nusa Tenggara Timur (Kupang) hanya
**1.407 mm/tahun**. Lima provinsi terkering membentuk satu sabuk di bagian selatan-tenggara: NTT, NTB,
Jawa Timur, Papua Selatan (Merauke), dan Banten. Sebaliknya, provinsi terbasah tersebar di Maluku Utara, Papua Tengah
(Nabire), Papua Barat Daya (Sorong), Sulawesi Barat, dan Sumatera Barat (Visualisasi 1).
**Implikasi:** kebijakan air dan pertanian tidak bisa disamaratakan. Wilayah sabuk tenggara perlu prioritas
penampungan air, sedangkan wilayah terbasah lebih membutuhkan mitigasi banjir dan longsor.

### Insight 2: Indonesia punya setidaknya tiga pola musim yang berbeda
- **Monsun tegas (Jawa, Bali–Nusa Tenggara):** Bali–Nusa Tenggara menerima **75,7%** hujan tahunannya pada
  November–Maret dan hanya **2,8%** pada Juli–Agustus. Untuk Jawa angkanya 64,9% dan 4,7%.
- **Ekuatorial/merata (Sumatera, Kalimantan, Sulawesi, Papua):** porsi November–Maret hanya 42–49%, dan
  Juli–Agustus masih menyumbang 12–16% hujan.
- **Lokal/terbalik (Maluku):** puncak hujan justru **Mei**, dan Juli–Agustus menyumbang 17,1%, tertinggi
  dari semua pulau, saat Jawa sedang kemarau.

(Visualisasi 2.) **Implikasi:** istilah "musim kemarau Juli–Agustus" hanya berlaku tegas di Jawa dan
Nusa Tenggara. Kalender tanam dan jadwal kegiatan luar ruang harus disesuaikan per wilayah.

### Insight 3: 2023 adalah tahun kering yang serentak secara nasional
Rata-rata hujan per provinsi pada 2023 hanya **2.542 mm**, turun sekitar **20%** dibanding tahun lain
(3.155–3.290 mm). **36 dari 38 provinsi** lebih kering dari rata-rata 5 tahunnya, dengan median anomali
**−19,4%**. Dampak terbesar terjadi di Jawa: hujan Juli–Oktober 2023 hanya **139 mm**, padahal pada 2022
mencapai **712 mm** (turun 80%). Penyimpangan serentak seperti ini bertepatan dengan fenomena El Niño 2023,
sedangkan 2022 yang lebih basah dari rata-rata (median anomali +9,4%) bertepatan dengan La Niña
(Visualisasi 3 dan 4).
**Implikasi:** variabilitas antartahun cukup besar untuk mengubah ketersediaan air secara nasional,
sehingga sinyal iklim global perlu menjadi bagian dari perencanaan musim tanam.

### Insight 4: Separuh hujan turun hanya dalam 6 jam di siang–sore hari
Di **semua** pulau, hujan memuncak pukul **14.00–15.00 waktu lokal** dan paling sedikit menjelang pagi.
Rentang 12.00–17.59 hanya mencakup 25% jam dalam sehari, tetapi menyumbang **51,2%** total hujan. Kontras
siklus harian paling kuat terjadi di Sulawesi (puncak **9,2×** jam terendahnya) dan Jawa (**7,1×**)
(Visualisasi 5). Pola ini khas hujan **konvektif**: pemanasan matahari pagi memicu awan hujan pada siang
hingga sore hari.
**Implikasi:** kegiatan luar ruang, pengeringan hasil panen, dan pekerjaan konstruksi lebih aman dijadwalkan
pada pagi hari.

### Insight 5: Sejak pertengahan 2023 suhu terus berada di atas normal
**Ke-18 bulan** sejak Juli 2023 lebih hangat dari rata-rata bulan kalender yang sama, dengan anomali tertinggi
**+0,54 °C** pada Februari 2024. Suhu rata-rata tahunan naik dari **26,9 °C (2022)** menjadi **27,5 °C (2024)**.
Hal ini konsisten dengan temuan notebook 02 bahwa 182 dari 345 outlier suhu terjadi pada 2023
(Visualisasi 6).
**Implikasi:** 2023–2024 merupakan periode panas yang berkelanjutan, bukan lonjakan sesaat. Ini relevan untuk
kebutuhan energi pendingin dan kesehatan masyarakat. Periode 5 tahun terlalu pendek untuk menyimpulkan
tren jangka panjang.

### Insight 6: Ketinggian, bukan lintang, yang menentukan suhu rata-rata
Korelasi suhu rata-rata dengan tekanan permukaan (proksi ketinggian) sebesar **r = 0,9**, sedangkan dengan
jarak dari khatulistiwa hanya **r = −0,1**. Wamena (±840 hPa) rata-rata **20,3 °C** dan Bandung (±935 hPa)
**24,1 °C**. Tanpa kedua kota dataran tinggi itu, suhu 36 ibu kota lainnya hanya berkisar
**26,3–28,5 °C** (Visualisasi 7).
**Implikasi:** di negara tropis, peta suhu sebaiknya dikoreksi dengan elevasi. Perbandingan suhu antarkota
tanpa memperhitungkan ketinggian bisa menyesatkan.

## 7. Keterbatasan

- Data adalah reanalisis ERA5 (grid ±25 km), bukan observasi stasiun. Hujan ekstrem lokal cenderung
  lebih rendah dibanding catatan alat ukur hujan.
- Satu titik per provinsi (ibu kota) tidak mewakili variasi di dalam provinsi, terutama provinsi yang luas
  dan bergunung.
- Periode 5 tahun cukup untuk pola musiman dan diurnal, tetapi belum cukup untuk menyimpulkan tren iklim
  jangka panjang.